# SQL mini cheatsheet

# Содержание

## 1. CREATE TABLE 

## 2. SELECT 

### 2.1. В общем виде. 

### 2.2. оформляющие конструкции (AS, CASE, distinct)

### 2.3. group by и агрегирующие функции. Where vs Having

### 2.4. округление

## 3. JOIN

### 3.1. биективное соединение по ключу

### 3.2. left/right/outer/inner join

## 4. ОКОННЫЕ ФУНКЦИИ

# 1. CREATE TABLE 

**создание таблицы**

CREATE TABLE customers (

    customer_id   INT PRIMARY KEY,
    
    customer_name  VARCHAR(100) NOT NULL,
        
    signup_date    DATE NOT NULL,

);

типы переменных INT, VARCHAR(100), DATE,...

**добавление**

INSERT INTO customers (customer_name, customer_id) VALUES ('Аня', 500), 
('Лена', 1500), 
('Вася', 25000);

**удаление**

DELETE FROM customers;

DELETE FROM customers WHERE customer_id = 500;


# 2. Select 

**2.1. В общем виде. Константы. Null

SELECT ...

FROM ... / JOIN ...

WHERE ...

GROUP BY ... 

HAVING ...

ORDER BY ... DESC/ASC

LIMIT ...;

SELECT * from...; выводит все 

SELECT CURRENT_DATE;

**NULL**

NULL нельзя проверять через = NULL или <> NULL; используются IS NULL и IS NOT NULL

COALESCE(value, replacement) — замена NULL;

NULLIF(a, b) — превращает a в NULL, если a = b;

**2.2. оформляющие конструкции (AS, CASE, distinct)**

AS — алиас столбца или таблицы;

CASE — условная логика; CASE WHEN X1 THEN Y1 WHEN X2 THEN Y2... ELSE YN END

DISTINCT — удаление дублей в проекции (но не исправление ошибок JOIN, см. далее)

SELECT DISTINCT
    product_name,
    price,
    price * 1.20 AS price_with_tax,
    CASE
        WHEN price < 1000 THEN 'budget'
        WHEN price < 5000 THEN 'standard'
        ELSE 'premium'
    END AS price_segment
FROM products;

**2.3. group by и агрегирующие функции**

COUNT() — считает число строк в группе. SUM() — считает сумму чисел в столбце. AVG() — считает среднее значение. MIN() / MAX()

! игнорируют NULL

! where используется для изначальных значений, having - для агрегирующих функций

SELECT 
    betting_type,
    ROUND(AVG(bet_amount), 2) AS avg_amount
FROM 
    bets
GROUP BY 
    betting_type
ORDER BY 
    avg_amount ASC;
    
select user_id, count(transaction_id) as transactions_count 
from transactions WHERE transaction_date >= '2024-01-01' 
  AND transaction_date < '2024-02-01'
  group by user_id
  Having count(transaction_id) > 5
  Order by user_id ASC

**2.4. округление**

ROUND(AVG(amount), 2) 

FLOOR(5.8)

RIGHT(column_name, 3) = 'abc';

c.name LIKE '%Salzburg%'


# 3. join

**3.1. биективное соединение по ключу**



Пример 1.

SELECT Countries.name AS country_name

FROM Cities 

JOIN Regions  ON Cities.regionid=Regions.id

JOIN Countries  ON Countries.id=Regions.countryid

WHERE Cities.name = 'Salzburg';

Пример 2.

SELECT e.name,

COALESCE(SUM(o.amount), 0) AS total_amount 

FROM employees e

LEFT JOIN (

select * from orders WHERE created_at >= '2024-01-01'

AND created_at < '2025-01-01') as o 

ON  e.id = o.employee_id

GROUP BY e.id, e.name

Пример 3.

SELECT

    o.order_id,

    o.order_date,

    c.customer_name,

    p.product_name,

    oi.quantity,

    oi.quantity * oi.unit_price AS line_amount

FROM orders o

JOIN customers c ON c.customer_id = o.customer_id

JOIN order_items oi ON oi.order_id = o.order_id

JOIN products p ON p.product_id = oi.product_id;

**3.2. left/right/outer/inner/cross join**

Визуализация всех типов соединения

https://github.com/fufaevvlvl/CheatSheets/blob/main/A_Join_Visualization.ipynb

# оконные функции

function(...) OVER ( #задаёт «окно» 

    PARTITION BY ... #делит это окно

    ORDER BY ... #совсем другая история: он задаёт порядок, в котором функция будет обходить строки раздела.
    
    ROWS BETWEEN ...
)

Оконная функция в SQL состоит из пяти ключевых элементов, каждый из которых строго определяет её поведение.

1. function - Сама функция (например, SUM, ROW_NUMBER, LAG) задаёт тип вычисления — агрегатное, ранжирующее или смещения, и применяется не к группе в целом, а к динамическому набору строк, связанному с текущей записью.

2. Ключевое слово OVER() является обязательным маркером оконной конструкции и отделяет вычисления от обычных скалярных операций, при этом пустые скобки означают окно из всех строк результирующего набора.

3. Предложение PARTITION BY разбивает строки на независимые логические группы (партиции), внутри которых функция перезапускается, — аналогично GROUP BY, но без свёртки, так что каждая строка сохраняется, а расчёт ведётся в пределах своей партиции.

4. Предложение ORDER BY внутри OVER устанавливает порядок строк внутри партиции (или всего набора), что критически важно для ранжирующих функций, а для агрегатов включает режим накопительного итога по умолчанию.

5. Опциональная рамка (ROWS | RANGE BETWEEN ...), задаваемая после ORDER BY, явно ограничивает множество соседних строк, участвующих в вычислении для текущей строки, позволяя реализовать скользящие средние, кумулятивные суммы с границами или сравнения с фиксированным смещением.

нумеруем сотрудников по убыванию зарплаты, причём отдельно для каждого отдела:

SELECT

    name,
    
    department,
    
    salary,
    
    RANK() OVER (PARTITION BY department ORDER BY salary DESC) AS dept_rank

FROM employees;

Накопительный итог

SELECT 

    date,
    
    sales,
    
    SUM(sales) OVER (ORDER BY date) AS total
    
    AVG(sales) OVER (ORDER BY date ROWS BETWEEN 2 PRECEDING AND CURRENT ROW) AS 3_day_av
    
FROM daily_sales    

select distinct user_id, FIRST_VALUE(item) OVER (PARTITION BY user_id ORDER BY transaction_ts ASC ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING) as item from Transactions   